# Full evaluation notebook (Kaggle T4 x2) -- A6 + A6.5

Runs the *full* (unsampled) benchmarks against both `d4` and `d6` SFT checkpoints, on GPU,
instead of the small local CPU spot-checks:

- `scripts/chat_eval.py`: ARC-Easy, ARC-Challenge, MMLU, GSM8K, HumanEval -- upstream nanochat's
  own benchmark suite. Expected near-baseline at this scale (see README/RESEARCH_LOG), run anyway
  for completeness.
- `scripts/eval_blimp.py`: BLiMP (Warstadt et al. 2020), 67 grammar categories x 1000 minimal
  pairs each -- a much better fit for what a model this size can plausibly do well on. Local
  spot-checks (3 categories x 30 pairs) already showed 73-93%, well above the 50% chance level.

Only GPU 0 is used for nothing here -- these are single-process, no `torchrun`/DDP needed (unlike
training), so this notebook doesn't split work across both T4s. That's fine; the point of T4 x2
here is just "whichever GPU accelerator is available", not parallelism.

Rough time budget (estimate, not measured yet): full `chat_eval.py` for one model is dominated by
the generative tasks (GSM8K ~1319 problems, HumanEval ~164, evaluated one at a time) -- maybe
45-70 min per model. Full `eval_blimp.py` (67 x 1000 pairs, batched) -- maybe 20-45 min per model.
Both models: **rough total 3-4 hours**. Cut `--max-problems`/`--max-pairs` down if that's too much
for the remaining GPU-hours budget -- see the commented-out fast alternative in each cell.

Upload via File -> Upload Notebook. Same 4 Kaggle Secrets as the training notebooks, T4 x2
accelerator, internet access (also needs to download ARC/MMLU/GSM8K/HumanEval/BLiMP from the HF
Hub on first run).

## Cell 1: clone repo, install dependencies

In [1]:
import os
import subprocess
import sys

REPO_URL = "https://github.com/nadeko0/nanochat-ru.git"
REPO_DIR = "/kaggle/working/repo"

if os.path.isdir(os.path.join(REPO_DIR, ".git")):
    print("Repo already present, pulling latest...")
    !git -C {REPO_DIR} pull
else:
    !git clone {REPO_URL} {REPO_DIR}

os.chdir(REPO_DIR)

def have(cmd):
    return subprocess.run(["bash", "-lc", f"command -v {cmd}"], capture_output=True).returncode == 0

if not have("uv"):
    !curl -LsSf https://astral.sh/uv/install.sh | sh
os.environ["PATH"] = f"{os.path.expanduser('~/.local/bin')}:{os.environ['PATH']}"

if not have("cargo"):
    !curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y
os.environ["PATH"] = f"{os.path.expanduser('~/.cargo/bin')}:{os.environ['PATH']}"

if not have("rclone"):
    !curl https://rclone.org/install.sh | sudo bash

!uv pip install --system --python {sys.executable} --extra gpu -r pyproject.toml

print("Cell 1 done.")

Cloning into '/kaggle/working/repo'...
remote: Enumerating objects: 210, done.
remote: Counting objects: 100% (210/210), done.
remote: Compressing objects: 100% (165/165), done.
remote: Total 210 (delta 93), reused 157 (delta 40), pack-reused 0 (from 0)
Receiving objects: 100% (210/210), 611.54 KiB | 5.01 MiB/s, done.
Resolving deltas: 100% (93/93), done.
info: downloading installer
warn: It looks like you have an existing rustup settings file at:
warn: /root/.rustup/settings.toml
warn: Rustup will install the default toolchain as specified in the settings file,
warn: instead of the one inferred from the default host triple.
info: profile set to default
info: default host triple is x86_64-unknown-linux-gnu
info: syncing channel updates for stable-x86_64-unknown-linux-gnu
info: latest update on 2026-07-16 for version 1.97.1 (8bab26f4f 2026-07-14)
info: downloading 6 components
        cargo downloading [#              ]   10.63 MiB (5.66 MiB/s, ETA: 2s)   
        cargo downloading [#  

## Cell 2: configure rclone, pull both d4 and d6 SFT checkpoints + tokenizer

In [2]:
import os
import subprocess
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
client_id = secrets.get_secret("GDRIVE_CLIENT_ID").strip()
client_secret = secrets.get_secret("GDRIVE_CLIENT_SECRET").strip()
oauth_token = secrets.get_secret("GDRIVE_OAUTH_TOKEN").strip()
folder_id = secrets.get_secret("GDRIVE_FOLDER_ID").strip()

rclone_conf_dir = os.path.expanduser("~/.config/rclone")
os.makedirs(rclone_conf_dir, exist_ok=True)
with open(os.path.join(rclone_conf_dir, "rclone.conf"), "w") as f:
    f.write(
        "[gdrive]\n"
        "type = drive\n"
        "scope = drive\n"
        f"client_id = {client_id}\n"
        f"client_secret = {client_secret}\n"
        f"token = {oauth_token}\n"
        f"root_folder_id = {folder_id}\n"
        "team_drive =\n"
    )

!rclone lsd gdrive:

DRIVE_REMOTE = "gdrive:"
NANOCHAT_BASE_DIR = "/kaggle/working/nanochat_cache"
os.environ["NANOCHAT_BASE_DIR"] = NANOCHAT_BASE_DIR
os.makedirs(NANOCHAT_BASE_DIR, exist_ok=True)

!rclone copy gdrive:tokenizer {NANOCHAT_BASE_DIR}/tokenizer --checksum -v
for tag in ["d4", "d6"]:
    !rclone copy gdrive:chatsft_checkpoints/{tag} {NANOCHAT_BASE_DIR}/chatsft_checkpoints/{tag} --checksum -v

print("Checkpoints ready:")
!ls {NANOCHAT_BASE_DIR}/chatsft_checkpoints/d4 {NANOCHAT_BASE_DIR}/chatsft_checkpoints/d6

           0 2026-08-10 16:16:45        -1 base_checkpoints
           0 2026-08-10 15:53:07        -1 base_data_climbmix
           0 2026-08-10 17:58:32        -1 chatsft_checkpoints
           0 2026-08-10 15:56:35        -1 tokenizer
2026/08/11 13:57:11 INFO  : token_bytes.pt: Copied (new)
2026/08/11 13:57:11 INFO  : tokenizer.pkl: Copied (new)
2026/08/11 13:57:11 INFO  : 
Transferred:   	  532.007 KiB / 532.007 KiB, 100%, 0 B/s, ETA -
Checks:                 0 / 0, -, Listed 2
Transferred:            2 / 2, 100%
Elapsed time:         0.9s

2026/08/11 13:57:12 INFO  : meta_000125.json: Copied (new)
2026/08/11 13:57:16 INFO  : optim_000125_rank0.pt: Copied (new)
2026/08/11 13:57:17 INFO  : model_000125.pt: Copied (new)
2026/08/11 13:57:17 INFO  : optim_000125_rank1.pt: Copied (new)
2026/08/11 13:57:17 INFO  : 
Transferred:   	  408.080 MiB / 408.080 MiB, 100%, 79.607 MiB/s, ETA 0s
Checks:                 0 / 0, -, Listed 4
Transferred:            4 / 4, 100%
Elapsed time:         5.

## Cell 3: full chat_eval.py -- d4

In [3]:
import os
os.chdir("/kaggle/working/repo")

# Full run, no -x limit. If this is taking too long / too much GPU budget, interrupt and rerun
# with e.g. `-x 200` (chat_eval.py's --max-problems flag) for a faster, still-informative sample.
!python -m scripts.chat_eval -i sft -g d4 2>&1 | tee /kaggle/working/chat_eval_d4.log

2026-08-11 13:58:17,633 - nanochat.common - INFO - Distributed world size: 1
2026-08-11 13:58:17,634 - nanochat.checkpoint_manager - INFO - Loading model from /kaggle/working/nanochat_cache/chatsft_checkpoints/d4 with step 125
2026-08-11 13:58:17,988 - nanochat.checkpoint_manager - INFO - Building model with config: {'sequence_len': 2048, 'vocab_size': 32768, 'n_layer': 4, 'n_head': 2, 'n_kv_head': 2, 'n_embd': 256, 'window_pattern': 'L'}
2026-08-11 13:58:18,834 - numexpr.utils - INFO - NumExpr defaulting to 4 threads.
Autodetected device type: cuda
Final: 602/2376 (25.34%)
ARC-Easy accuracy: 25.34%
Final: 265/1172 (22.61%)
ARC-Challenge accuracy: 22.61%
Final: 3215/14042 (22.90%)
MMLU accuracy: 22.90%
Rank 0 | 1/1319 (0.08%)
Final: 1/1319 (0.08%)
GSM8K accuracy: 0.08%
Rank 0 | 0/164 (0.00%)
Final: 0/164 (0.00%)
HumanEval accuracy: 0.00%
ChatCORE metric: -0.0109


## Cell 4: full chat_eval.py -- d6

In [4]:
import os
os.chdir("/kaggle/working/repo")

!python -m scripts.chat_eval -i sft -g d6 2>&1 | tee /kaggle/working/chat_eval_d6.log

2026-08-11 14:19:21,265 - nanochat.common - INFO - Distributed world size: 1
2026-08-11 14:19:21,265 - nanochat.checkpoint_manager - INFO - Loading model from /kaggle/working/nanochat_cache/chatsft_checkpoints/d6 with step 63
2026-08-11 14:19:21,736 - nanochat.checkpoint_manager - INFO - Building model with config: {'sequence_len': 2048, 'vocab_size': 32768, 'n_layer': 6, 'n_head': 3, 'n_kv_head': 3, 'n_embd': 384, 'window_pattern': 'L'}
2026-08-11 14:19:21,978 - numexpr.utils - INFO - NumExpr defaulting to 4 threads.
Autodetected device type: cuda
Final: 591/2376 (24.87%)
ARC-Easy accuracy: 24.87%
Final: 263/1172 (22.44%)
ARC-Challenge accuracy: 22.44%
Final: 3221/14042 (22.94%)
MMLU accuracy: 22.94%
Rank 0 | 0/1319 (0.00%)
Final: 0/1319 (0.00%)
GSM8K accuracy: 0.00%
Rank 0 | 0/164 (0.00%)
Final: 0/164 (0.00%)
HumanEval accuracy: 0.00%
ChatCORE metric: -0.0127


## Cell 5: full BLiMP eval -- d4

In [5]:
import os
os.chdir("/kaggle/working/repo")

# All 67 categories, 1000 pairs each, batched. If this is too slow, lower --max-pairs (e.g. 200).
!python -m scripts.eval_blimp -i sft -g d4 --batch-size 64 2>&1 | tee /kaggle/working/blimp_d4.log

2026-08-11 14:51:16,935 - nanochat.common - INFO - Distributed world size: 1
2026-08-11 14:51:16,935 - nanochat.checkpoint_manager - INFO - Loading model from /kaggle/working/nanochat_cache/chatsft_checkpoints/d4 with step 125
2026-08-11 14:51:17,278 - nanochat.checkpoint_manager - INFO - Building model with config: {'sequence_len': 2048, 'vocab_size': 32768, 'n_layer': 4, 'n_head': 2, 'n_kv_head': 2, 'n_embd': 256, 'window_pattern': 'L'}
2026-08-11 14:51:18,308 - numexpr.utils - INFO - NumExpr defaulting to 4 threads.
Autodetected device type: cuda
[1/67] adjunct_island: 68.2% (682/1000)
[2/67] anaphor_gender_agreement: 55.3% (553/1000)
[3/67] anaphor_number_agreement: 81.7% (817/1000)
[4/67] animate_subject_passive: 63.1% (631/1000)
[5/67] animate_subject_trans: 64.0% (640/1000)
[6/67] causative: 59.8% (598/1000)
[7/67] complex_NP_island: 45.1% (451/1000)
[8/67] coordinate_structure_constraint_complex_left_branch: 42.7% (427/1000)
[9/67] coordinate_structure_constraint_object_extract

## Cell 6: full BLiMP eval -- d6

In [7]:
import os
os.chdir("/kaggle/working/repo")

!python -m scripts.eval_blimp -i sft -g d6 --batch-size 64 2>&1 | tee /kaggle/working/blimp_d6.log

2026-08-11 14:53:52,454 - nanochat.common - INFO - Distributed world size: 1
2026-08-11 14:53:52,455 - nanochat.checkpoint_manager - INFO - Loading model from /kaggle/working/nanochat_cache/chatsft_checkpoints/d6 with step 63
2026-08-11 14:53:52,935 - nanochat.checkpoint_manager - INFO - Building model with config: {'sequence_len': 2048, 'vocab_size': 32768, 'n_layer': 6, 'n_head': 3, 'n_kv_head': 3, 'n_embd': 384, 'window_pattern': 'L'}
2026-08-11 14:53:53,191 - numexpr.utils - INFO - NumExpr defaulting to 4 threads.
Autodetected device type: cuda
[1/67] adjunct_island: 70.7% (707/1000)
[2/67] anaphor_gender_agreement: 55.2% (552/1000)
[3/67] anaphor_number_agreement: 91.7% (917/1000)
[4/67] animate_subject_passive: 68.5% (685/1000)
[5/67] animate_subject_trans: 79.7% (797/1000)
[6/67] causative: 60.8% (608/1000)
[7/67] complex_NP_island: 52.9% (529/1000)
[8/67] coordinate_structure_constraint_complex_left_branch: 56.9% (569/1000)
[9/67] coordinate_structure_constraint_object_extracti